## Pipeline de simulación de repertorios inmunológicos  ground truth con immuneSIM

Se utilizó el paquete **immuneSIM** en R para generar repertorios inmunológicos simulados bajo distintos escenarios biológicos y profundidades de secuenciación. Este enfoque permite modelar de forma controlada la estructura clonal de los repertorios de células B, incluyendo la asignación de genes V(D)J, la generación de secuencias de la región CDR3 y la distribución de abundancias clonales.

A diferencia de los flujos basados en inferencia (SHazaM + Change-O), immuneSIM entrega directamente la estructura del sistema simulado, permitiendo construir un **ground truth basado en abundancias clonales explícitas** para el análisis de diversidad.



### Generación del repertorio simulado

Los repertorios fueron generados utilizando la función `immuneSIM()`, la cual produce un objeto que contiene información a nivel de secuencia/clon, incluyendo:

- Secuencia nucleotídica (`sequence`)
- Secuencia aminoacídica (`sequence_aa`)
- Genes V, D y J (`v_call`, `d_call`, `j_call`)
- Región CDR3 (`junction`, `junction_aa`)
- Características de recombinación (`np1`, `np2`, `del_v`, `del_d_5`, `del_d_3`, `del_j`)
- Alineamientos V(D)J (`v_sequence_alignment`, `d_sequence_alignment`, `j_sequence_alignment`)
- Eventos de hipermutación somática (`shm_events`)
- Abundancia clonal (`counts`)
- Frecuencia relativa (`freqs`)
- Identificador del repertorio (`name_repertoire`)



### Construcción del archivo ground truth (.tsv)

A partir del objeto generado por immuneSIM, se construyó una tabla de trabajo que representa el **ground truth del repertorio simulado**, basada en la estructura interna del simulador.

En este enfoque, cada fila corresponde a una secuencia/clon simulado, cuya abundancia está definida por el simulador.

Las variables incluidas en el archivo fueron:

- `sequence` → secuencia nucleotídica  
- `sequence_aa` → secuencia aminoacídica  
- `v_call`, `d_call`, `j_call` → asignación de genes V(D)J  
- `junction`, `junction_aa` → región CDR3  
- `counts` → número de células asociadas a cada secuencia/clon  
- `freqs` → frecuencia relativa en el repertorio  



### Exportación del archivo

El dataset fue exportado en formato `.tsv` utilizando `write.table()`, generando un archivo por cada escenario biológico (Control, Naive, Sangre periférica, Folicular y Extrafolicular) y por cada profundidad de secuenciación (100 a 102,400 secuencias).



###  Uso del ground truth

Este archivo fue utilizado para:

- Cálculo directo de métricas de diversidad (riqueza clonal, Shannon, Simpson)  
- Análisis del efecto de la profundidad de secuenciación  
- Comparación con métodos de inferencia clonal basados en SHazaM + Change-O  



###  Consideración metodológica

El uso de immuneSIM permite disponer de una estructura clonal simulada con abundancias explícitas (`counts`), lo que facilita la evaluación de métricas de diversidad en un entorno controlado. Esto permite diferenciar claramente entre:

- **Ground truth (estructura simulada basada en abundancias)**  
- **Clustering inferido (Change-O + SHazaM)**  

## Configuración del entorno reproducible con renv

Se utilizó el paquete **renv** para gestionar un entorno 
reproducible del proyecto en R, permitiendo aislar y fijar 
las versiones exactas de los paquetes utilizados.

Este enfoque asegura que los análisis puedan ser replicados 
en otros sistemas sin problemas de compatibilidad.

### Inicialización del entorno

- `renv::init()` → inicializa el proyecto y genera el archivo `renv.lock`, 
  que almacena las versiones de los paquetes utilizados.

### Instalación de paquetes

Se instalaron paquetes desde:

- **CRAN**: `dplyr`, `ggplot2`, `viridisLite`, `readr`, `here`
- **Bioconductor**: `Biostrings`, `IRanges`, `GenomicRanges`

Estos paquetes fueron instalados usando `install.packages()` 
y `BiocManager::install()` según corresponda.

### Control de versiones

- `renv::snapshot()` → guarda las versiones exactas de los paquetes 
  en el archivo `renv.lock`.

- `renv::restore()` → permite restaurar el entorno en otro computador 
  con las mismas versiones de paquetes.

Este flujo garantiza la **reproducibilidad completa del análisis**.

## Simulación de repertorios con immuneSIM

Se generaron repertorios simulados de secuencias de inmunoglobulinas 
utilizando el paquete **immuneSIM**, configurando distintos parámetros 
que controlan la composición, diversidad y características biológicas 
de las secuencias.

### Descripción de variables

- **name_repertoire**: nombre del repertorio simulado  
- **number_of_seqs**: número total de secuencias generadas  
- **species**: especie (ej. humano)  
- **receptor**: tipo de receptor (`ig` para BCR, `tr` para TCR)  
- **chain**: tipo de cadena (pesada o liviana)  
- **verbose**: activación de mensajes durante la simulación  

### Parámetros de distribución clonal

- **equal_cc**: define si todos los clonotipos tienen el mismo tamaño  
- **user_defined_alpha**: controla la uniformidad de la distribución clonal  
  (valores altos generan mayor desigualdad entre clones)

### Mutaciones somáticas (SHM)

- **shm**: modelo de mutación somática. Puede tomar los valores:

  - `none`: no se simulan mutaciones  
  - `poisson`: mutaciones aleatorias sin sesgo  
  - `data`: basado en perfiles reales (más mutaciones en regiones CDR)  
  - `naive`: secuencias sin SHM (puede incluir artefactos técnicos)  
  - `motif`: mutaciones dirigidas por motivos específicos  

- **shm.prob**: probabilidad de mutación por secuencia  
  (ej. 15/350 ≈ 15 mutaciones en 350 nucleótidos)

### Parámetros de recombinación V(D)J

- **vdj_noise**: introduce variabilidad en la selección de genes V, D y J  
  (rango 0–1; valores más altos → mayor aleatoriedad)

### Longitud de CDR3

- **max_cdr3_length / min_cdr3_length**: límites para la longitud de CDR3  

En humanos, la longitud típica de CDR3 en cadenas pesadas (IgH) 
varía entre 10 y 25 aminoácidos, con una mediana aproximada 
de 15–16.

In [31]:
library(immuneSIM)
library(Biostrings)
library(dplyr)
library(here) 

In [32]:
n_seqs <- 100
scenario <- "D"

sim_repertoire <- immuneSIM(
  name_repertoire = paste0("sim_rep_", n_seqs),
 number_of_seqs = n_seqs,
  species = "hs",
  receptor = "ig",
  chain = "h",
  verbose= TRUE,
  equal_cc = FALSE,
  user_defined_alpha = 1.5,  
  shm = "motif",
  shm.prob = 40/350,
  vdj_noise = 0,
  max_cdr3_length =15, 
  min_cdr3_length =13,
  )


initializing sim..
simulated sequences: 10 
simulated sequences: 20 
simulated sequences: 30 
simulated sequences: 40 
simulated sequences: 50 
simulated sequences: 60 
simulated sequences: 70 
simulated sequences: 80 
simulated sequences: 90 
simulated sequences: 100 


In [33]:
df_gt <- as.data.frame(sim_repertoire)


In [34]:
gt_table <- df_gt %>%
  select(
    sequence,
    v_call,
    d_call,
    j_call,
    junction,
    counts,
    freqs
    )

In [35]:
gt_file <- file.path(
  "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/immunesim_ground_truth",
  paste0("gt_", scenario, "_", n_seqs, ".tsv")
)
write.table(
  gt_table,
  file = gt_file,
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

message("Guardado en: ", gt_file)

Guardado en: /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/immunesim_ground_truth/gt_D_100.tsv



In [36]:
names(sim_repertoire)

[1] "sequence"             "sequence_aa"          "junction"            
 [4] "junction_aa"          "v_call"               "d_call"              
 [7] "j_call"               "np1"                  "np2"                 
[10] "del_v"                "del_d_5"              "del_d_3"             
[13] "del_j"                "v_sequence_alignment" "d_sequence_alignment"
[16] "j_sequence_alignment" "freqs"                "counts"              
[19] "shm_events"           "name_repertoire"

In [37]:
nrow(sim_repertoire)
sum(sim_repertoire$counts)
head(sim_repertoire$counts)

[1] 100

[1] 2410

[1] 1000  354  192  125   89   68

In [38]:
head(sim_repertoire$counts)

[1] 1000  354  192  125   89   68